# Quotas, retries, service tiers and latency

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

What keeps `bedrock-mantle` calls working under load: how throughput is metered, how to
retry, which service tier to ask for, and how to measure latency. Authentication and URL
paths are in
[`01-endpoints-auth-and-paths`](01-endpoints-auth-and-the-three-paths.ipynb).

**Covered:** the quota model, retries with backoff, service tiers, time to first token,
and a client that puts these together.

## Setup

The notebook uses the OpenAI SDK against Gemma 4, which `bedrock-mantle` serves in every
Region it runs in, and boto3 for Service Quotas. Section 3 adds GPT-6 Astra, because the
set of service tiers differs between the two. Astra is on `bedrock-mantle` in `us-east-1`
and `us-west-2`, and `us-east-1` is the Region that serves `ultrafast`, so section 3 is
written for `us-east-1`. In `us-west-2` the last row of its table reads `not in its list`
for both models.

In [1]:
import json
import statistics
import time
from datetime import timedelta

import boto3
from aws_bedrock_token_generator import provide_token
from openai import BadRequestError, OpenAI

REGION = "us-east-1"
MODEL = "google.gemma-4-31b"
# A second model for section 3, which serves a tier that Gemma 4 does not.
ASTRA = "openai.gpt-6-astra"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws/openai/v1"
client = OpenAI(base_url=BASE_URL, api_key=provide_token(region=REGION), timeout=120)

## 1. The quota model

Quotas on `bedrock-mantle` are metered in tokens, with no requests-per-minute quota.

| | `bedrock-runtime` | `bedrock-mantle` |
|---|---|---|
| Token quota | one tokens-per-minute (TPM) quota | separate input TPM and output TPM |
| Request quota | requests per minute | none |
| Raising a quota | Service Quotas | an AWS Support case |

The two quotas are checked at different moments, which is why the output budget you ask
for is an input-side concern. At admission the service reserves your input tokens plus
that budget against the input TPM quota and returns 429 when the reservation does not
fit, then releases the unused part when the call ends. Output tokens are counted against
the output TPM quota as they are generated, so the budget does not hold output quota in
advance; as of October 2026 a request whose budget exceeded the whole output TPM quota
for the model was still admitted and completed. Running out of output TPM during
generation is not a 429: the
[User Guide](https://docs.aws.amazon.com/bedrock/latest/userguide/quotas-mantle.html)
says generation stops and the reply is returned with a finish reason that marks the
cutoff, so check the status on every reply rather than waiting for an exception.

The parameter is named per API: `max_output_tokens` on the Responses API and
`max_tokens` on Chat Completions. The Responses API takes `max_tokens` with a 200 and
ignores it, as it does other keys it does not recognise, so a request that sets it has
no budget at all and the reply carries `max_output_tokens` of `null`.

Leaving the budget unset reserves the model's maximum, which is the largest reservation
you can make, so set it and keep it close to what you need. A value far above what you
use holds input TPM you do not spend and can throttle you under load. Cached input
tokens do not count against input TPM. Only some models have a per-account quota in
Service Quotas; the rest share internal capacity, which is why the retry in section 2
matters.

In [2]:
quotas = [(q["QuotaName"], q["Value"])
          for page in boto3.client("service-quotas", region_name=REGION).get_paginator("list_service_quotas")
          .paginate(ServiceCode="bedrock")
          for q in page["Quotas"] if "mantle" in q["QuotaName"].lower()]
print(f"{len(quotas)} bedrock-mantle quotas in Service Quotas, for example:")
for name, value in sorted(quotas)[:5]:
    print(f"  {value:>14,.0f}  {name}")

22 bedrock-mantle quotas in Service Quotas, for example:
       2,000,000  [bedrock-mantle endpoint] Input tokens per minute for Claude Fable 5
       5,000,000  [bedrock-mantle endpoint] Input tokens per minute for Claude Fable 5.1
      20,000,000  [bedrock-mantle endpoint] Input tokens per minute for Claude Opus 4.7
      20,000,000  [bedrock-mantle endpoint] Input tokens per minute for Claude Opus 4.8
      20,000,000  [bedrock-mantle endpoint] Input tokens per minute for Claude Opus 5


## 2. Retries with backoff

Treat 429 (throttled) and 5xx (capacity) as transient and retry them with exponential
backoff and jitter; other 4xx errors are a bug in the request, so fail fast. The OpenAI
SDK retries 408, 409, 429 and 5xx itself, twice by default, with backoff, and
`max_retries` raises that. Code that calls the endpoints without the SDK needs its own
loop, like `post` in `_shared/bedrock.py`. A bad request comes back at once.

In [3]:
patient = client.with_options(max_retries=5)
reply = patient.responses.create(model=MODEL, input="Reply with exactly: OK", max_output_tokens=16, store=False)
print("with max_retries=5:", reply.output_text.strip())

start = time.perf_counter()
try:
    patient.responses.create(model=MODEL, input="Reply OK", max_output_tokens=8, store=False)  # below the minimum of 16
except BadRequestError as exc:
    print(f"a 400 is not retried: HTTP {exc.status_code} after {time.perf_counter() - start:.1f}s")

with max_retries=5: OK
a 400 is not retried: HTTP 400 after 0.1s


## 3. Service tiers

`service_tier` trades latency against price: `priority` for latency-critical traffic at
a premium, `default` for everyday work, and `flex` at a discount for work that can wait,
such as evaluations. `auto` lets the service choose. The reply says which tier served
the request, so log that value rather than the one you asked for.

Which tiers you can ask for depends on the model and the Region, not on the endpoint.
GPT-6 Astra takes a fifth value, `ultrafast`, on `bedrock-mantle` in `us-east-1`,
announced as a premium speed tier for latency-sensitive work. The cell below shows it
accepted and served; it does not measure latency, which section 4 covers. The same model
in `us-west-2` does not take the value at all, and GPT-6 Sol, GPT-6.1 Sol and Gemma 4 in
`us-east-1` refuse it (September 2026). Nor is the tier a property of the endpoint:
`us.openai.gpt-6-astra` on `bedrock-runtime` in `us-east-1` answered with `service_tier:
ultrafast` as well, and `us.openai.gpt-6-sol` there refused the value with the same 400
as on `bedrock-mantle` (October 2026). Ask for `ultrafast` explicitly, because on Astra
both `auto` and an omitted `service_tier` were served by `default`.

A tier you cannot have comes back as one of two 400s, and which one says where to look. A
value outside the model's list returns `Invalid value: 'ultrafast'. Supported values are:
'auto', 'default', 'flex', and 'priority'.`, and that list differs by model and Region, so
one model's error does not tell you what another will take. That list is what the request
validator takes rather than what the model will serve: in `us-east-1` GPT-6 Sol lists
`ultrafast` among its supported values and still refuses it with the second 400 (October
2026). A value the model knows but will not serve returns `'ultrafast' is not supported
for 'service_tier' on this model`.
`reserved` is not a request parameter: it returns 400 `Invalid value: 'reserved'`, because
reserved capacity is arranged with your account team.

In [4]:
TIERS = ("auto", "default", "flex", "priority", "ultrafast")


def served_tier(model: str, tier: str) -> str:
    """The tier that served the request, or which kind of 400 refused it."""
    try:
        reply = client.responses.create(model=model, input="Reply OK", max_output_tokens=16,
                                        service_tier=tier, store=False)
        return f"served by {reply.service_tier}"
    except BadRequestError as exc:
        # Two refusals worth telling apart: `invalid_value` means the value is not in this
        # model's list, and the message names the values that are, while `validation_error`
        # means the model knows the value and will not serve it.
        return "not in its list" if exc.code == "invalid_value" else "refused by the model"


print(f"{'asked for':11} {MODEL:22} {ASTRA}")
for tier in TIERS:
    print(f"{tier:11} {served_tier(MODEL, tier):22} {served_tier(ASTRA, tier)}")

asked for   google.gemma-4-31b     openai.gpt-6-astra


auto        served by default      served by default


default     served by default      served by default


flex        served by flex         refused by the model


priority    served by priority     refused by the model


ultrafast   not in its list        served by ultrafast


## 4. Time to first token

Time to first token is mostly prefill, the model reading your prompt, plus queueing, so
longer prompts start later and prompt caching shortens the wait. Tiers change queue
priority: `flex` is served after other traffic and `priority` ahead of it, so under
contention `flex` starts later and `priority` sooner. On a quiet account the gaps are
small and change from run to run, and a `flex` reply can arrive in one burst, with its
first token close to the end. The cell takes the median of three calls per tier, which
is one sample, not a benchmark. If you measure the effect of prompt length yourself,
vary the text, because a repeated filler sentence is a cacheable prefix and measures the
cache.

In [5]:
DEADLINE = 120  # seconds for one call, so a queued flex request cannot hold the cell


def first_token(tier: str) -> tuple[float, float]:
    """Seconds to the first text delta, and to the end of the stream."""
    start, first = time.perf_counter(), None
    # Two bounds, because one is not enough: `timeout` ends a read that stalls, and the
    # check inside the loop ends a stream that trickles bytes for longer than DEADLINE.
    with client.responses.stream(model=MODEL, service_tier=tier, max_output_tokens=300, timeout=60,
                                 input="Explain what a distributed inference engine does, in three sentences.",
                                 store=False) as stream:
        for event in stream:
            if event.type == "response.output_text.delta" and first is None:
                first = time.perf_counter() - start
            if time.perf_counter() - start > DEADLINE:
                print(f"  {tier}: gave up after {DEADLINE}s")
                break
    return first or float("nan"), time.perf_counter() - start


print(f"{'tier':9} {'first token p50':>16} {'total p50':>10}")
for tier in ("default", "flex", "priority"):
    runs = [first_token(tier) for _ in range(3)]
    print(f"{tier:9} {statistics.median(r[0] for r in runs):>15.2f}s {statistics.median(r[1] for r in runs):>9.2f}s")

tier       first token p50  total p50


default              0.37s      1.14s


flex                 1.26s      1.33s


priority             0.52s      1.35s


## 5. A client that puts it together

A starting point for your own client, with a key replaced before it expires, a timeout,
retries, a service tier, `store=False` to opt out of retention, and the served tier
recorded with each result. A minted key cannot be refreshed and lasts at most 12 hours,
so a client that mints one in its constructor and holds it stops working while it runs:
once the key has expired the call returns 401 `invalid_api_key` with `Signature expired:
… is now earlier than …`, measured on `bedrock-mantle` in `us-east-1` in October 2026.
This one mints a replacement as the key it holds nears expiry and assigns it to the
existing client, which reads `api_key` on every request. For several threads sharing one
client, or several clients sharing one key, use the thread-safe provider in
[`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb). It has had no
security review; adapt it to your own error handling and observability. Raise load
gradually over minutes rather than in one step, because a sudden jump from zero is how
requests get shed.

In [6]:
class MantleResponses:
    """A thin wrapper over the Responses API with the settings this notebook recommends."""

    KEY_TTL = timedelta(minutes=15)  # a key is minted for this long, not the 12-hour maximum

    def __init__(self, model: str, tier: str = "default", region: str = REGION):
        self.model, self.tier, self.region = model, tier, region
        self._token, self._replace_after = None, -float("inf")  # -inf: a key is due now
        self.client = OpenAI(base_url=f"https://bedrock-mantle.{region}.api.aws/openai/v1",
                             api_key=self._key(), timeout=120, max_retries=5)

    def _key(self) -> str:
        """The held key, or a freshly minted one once the held key is close to expiry."""
        # monotonic, because this is an elapsed duration and a wall clock can step.
        if time.monotonic() >= self._replace_after:
            self._token = provide_token(region=self.region, expiry=self.KEY_TTL)
            self._replace_after = time.monotonic() + self.KEY_TTL.total_seconds() - 120
        return self._token

    def ask(self, prompt: str, max_output_tokens: int = 300) -> dict:
        # The SDK reads api_key on every request, so a replacement needs no new client.
        self.client.api_key = self._key()
        reply = self.client.responses.create(model=self.model, input=prompt, service_tier=self.tier,
                                             max_output_tokens=max_output_tokens, store=False)
        if reply.status != "completed":
            raise RuntimeError(f"status {reply.status}: raise max_output_tokens")
        return {"text": reply.output_text, "tier": reply.service_tier, "usage": reply.usage.model_dump()}


result = MantleResponses(MODEL, tier="flex").ask("What is the capital of Australia? Answer in one word.")
print(result["text"].strip(), "| served by", result["tier"], "|", json.dumps(result["usage"]))

Canberra | served by flex | {"input_tokens": 45, "input_tokens_details": {"cache_write_tokens": 0, "cached_tokens": 0}, "output_tokens": 3, "output_tokens_details": {"reasoning_tokens": 0}, "total_tokens": 48}


## Things that differ

What trips people up here, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Throttling | 429 with no requests-per-minute quota to raise | Retry with backoff, and keep `max_output_tokens` close to what you need (section 1). |
| Capacity | 503 or `CapacityExceeded` after a sudden rise in traffic | Retry with backoff and raise load over minutes. |
| Quota increases | No Service Quotas entry for most models | Open an AWS Support case. |
| `reserved` tier | 400 `Invalid value: 'reserved'` | Arrange reserved capacity with your account team. |
| `ultrafast` tier | On Gemma 4, 400 `Invalid value: 'ultrafast'. Supported values are: 'auto', 'default', 'flex', and 'priority'.`; on GPT-6 Sol, 400 `'ultrafast' is not supported for 'service_tier' on this model` | Ask for it only where it is served: GPT-6 Astra in `us-east-1`, on either endpoint, as of October 2026 (section 3). |
| The tier that served you | `service_tier` in the reply can differ from the request | Log the served tier (section 3). |
| `max_tokens` on the Responses API | 200, with `max_output_tokens` of `null` in the reply and the answer running to the model's limit | Use `max_output_tokens`; `max_tokens` is the Chat Completions name (section 1). |
| `max_output_tokens` below 16 | 400 `Invalid 'max_output_tokens': integer below minimum value` | Ask for at least 16. |

## Next

- Converse and inference profiles on `bedrock-runtime`:
  [`04-bedrock-runtime-converse-and-profiles`](04-bedrock-runtime-converse-and-profiles.ipynb)
- Production hardening across families:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)
- Then the notebook for your model family, such as
  [`03-google-gemma/01`](../03-google-gemma/01-gemma4-end-to-end.ipynb)